# Part 2, Task 1 — Data Pipeline Construction

This notebook is an **annotated walkthrough of `pipeline.py`** — it imports
and calls the exact same functions defined there (nothing is re-implemented
or duplicated here), so the logic you see running is identical to what
`python pipeline.py` or `main.py` runs. The point of this notebook is to
show each cleaning step's effect on the data individually, with output
visible inline, rather than as one opaque script run.

Run `pipeline.py` (directly, or via `main.py`) is still the way this
project is actually meant to be executed end-to-end / in the CLI app —
this notebook is a companion for inspecting *how* it works.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Locate the part2_python directory (containing pipeline.py) regardless of
# where Jupyter's working directory happens to be when this runs.
_candidates = [Path.cwd(), Path.cwd() / "part2_python", Path.cwd().parent, Path.cwd().parent / "part2_python"]
for _c in _candidates:
    if (_c / "pipeline.py").exists():
        sys.path.insert(0, str(_c))
        break

import pipeline

pipeline.configure_logging("INFO")
print(f"Raw input:  {pipeline.DEFAULT_INPUT}")
print(f"Cleaned output: {pipeline.DEFAULT_OUTPUT}")

Raw input:  C:\Users\farid\Course\Week 33\Capstone project\smart-city-traffic-capstone\smart-city-traffic-capstone\Metro_Interstate_Traffic_Volume.csv
Cleaned output: C:\Users\farid\Course\Week 33\Capstone project\smart-city-traffic-capstone\smart-city-traffic-capstone\part2_python\cleaned_traffic_data.csv


## Load the raw data

In [2]:
df_raw = pipeline.load_raw_data(pipeline.DEFAULT_INPUT)
print(f"\nShape: {df_raw.shape}")
df_raw.head()

2026-09-15 11:46:25 | INFO     | pipeline | Successfully loaded raw data from Metro_Interstate_Traffic_Volume.csv: 48204 rows, 9 columns

Shape: (48204, 9)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


## Validate schema

Runs before any other processing — raises `SchemaValidationError` immediately
if an expected column is missing, rather than letting downstream steps fail
confusingly later.

In [3]:
pipeline.validate_schema(df_raw)
print("Schema OK — safe to proceed.")

2026-09-15 11:46:25 | INFO     | pipeline | Schema validation passed — all 9 expected columns present
Schema OK — safe to proceed.


## Clean the data, one step at a time

`clean_data()` in `pipeline.py` runs these four steps in sequence; they're
split out here so each one's effect is visible on its own.

### 1. Standardise categorical values
Fixes inconsistent casing in `weather_description` and makes the `holiday`
column's "not a holiday" case explicit via a new `is_holiday` boolean.

In [4]:
df1 = pipeline.standardise_categorical_values(df_raw)
df1[["weather_description", "holiday", "is_holiday"]].head()

2026-09-15 11:46:25 | WARNING  | pipeline | Standardised casing/whitespace on 1730 rows in 'weather_description' (reason: inconsistent capitalisation, e.g. 'Sky is Clear' vs 'sky is clear')
2026-09-15 11:46:25 | INFO     | pipeline | Standardised 'holiday' column: 61 rows flagged as a public holiday, 48143 rows are ordinary days (added boolean 'is_holiday' column)


,weather_description,holiday,is_holiday
0,scattered clouds,<NA>,False
1,broken clouds,<NA>,False
2,overcast clouds,<NA>,False
3,overcast clouds,<NA>,False
4,broken clouds,<NA>,False


### 2. Parse and validate `date_time`
Converts to a real datetime dtype and drops any row whose timestamp can't
be parsed.

In [5]:
df2 = pipeline.parse_and_validate_datetime(df1)
print(f"Rows before: {len(df1):,}  ->  after: {len(df2):,}")
df2["date_time"].head()

2026-09-15 11:46:25 | INFO     | pipeline | Parsed date_time to datetime dtype for 48204 rows (from 48204 originally)
Rows before: 48,204  ->  after: 48,204


0   2012-10-02 09:00:00
1   2012-10-02 10:00:00
2   2012-10-02 11:00:00
3   2012-10-02 12:00:00
4   2012-10-02 13:00:00
Name: date_time, dtype: datetime64[ns]

### 3. Remove duplicate rows
Drops exact duplicate rows, then collapses rows that share a timestamp
(multiple weather observations logged for the same hour) down to one row
per hour.

In [6]:
df3 = pipeline.remove_duplicate_rows(df2)
print(f"Rows before: {len(df2):,}  ->  after: {len(df3):,}  (removed {len(df2) - len(df3):,})")

2026-09-15 11:46:25 | WARNING  | pipeline | Removed 17 fully duplicate rows (reason: identical row appeared more than once in the raw file)
2026-09-15 11:46:25 | WARNING  | pipeline | Collapsed 7612 rows that shared a timestamp with another row down to one row per hour (reason: the weather API logged more than one simultaneous condition for the same hour; the first reported condition for each hour was kept)
Rows before: 48,204  ->  after: 40,575  (removed 7,629)


### 4. Handle outliers (impossible sensor readings)
`temp <= 0K` and `rain_1h > 9000mm` are physically impossible — both get
imputed using that month's median of the *valid* readings, computed
explicitly month-by-month so winter/summer values don't bleed into each
other.

In [7]:
print("Before handle_outliers():")
print(f"  Rows with temp <= 0K:        {(df3['temp'] <= pipeline.IMPOSSIBLE_TEMP_K).sum()}")
print(f"  Rows with rain_1h > 9000mm:  {(df3['rain_1h'] > pipeline.MAX_PLAUSIBLE_RAIN_MM).sum()}")

df4 = pipeline.handle_outliers(df3)

print("\nAfter handle_outliers():")
print(f"  Rows with temp <= 0K:        {(df4['temp'] <= pipeline.IMPOSSIBLE_TEMP_K).sum()}")
print(f"  Rows with rain_1h > 9000mm:  {(df4['rain_1h'] > pipeline.MAX_PLAUSIBLE_RAIN_MM).sum()}")

Before handle_outliers():
  Rows with temp <= 0K:        10
  Rows with rain_1h > 9000mm:  1
2026-09-15 11:46:25 | WARNING  | pipeline | Imputed 10 rows with an impossible temperature (<= 0K) using that month's median temperature (reason: 0K/absolute-zero sensor error)
2026-09-15 11:46:25 | WARNING  | pipeline | Imputed 1 rows with an implausible rain_1h value (> 9000mm in one hour) using that month's median rainfall (reason: sensor error — no real hourly rainfall reaches this level)

After handle_outliers():
  Rows with temp <= 0K:        0
  Rows with rain_1h > 9000mm:  0


## Save the cleaned dataset

In [8]:
pipeline.save_cleaned_data(df4, pipeline.DEFAULT_OUTPUT)
print(f"\nFinal cleaned shape: {df4.shape}")
df4.head()

2026-09-15 11:46:25 | INFO     | pipeline | Saved cleaned dataset to C:\Users\farid\Course\Week 33\Capstone project\smart-city-traffic-capstone\smart-city-traffic-capstone\part2_python\cleaned_traffic_data.csv (40575 rows, 10 columns)

Final cleaned shape: (40575, 10)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume,is_holiday
0,<NA>,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545,False
1,<NA>,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516,False
2,<NA>,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767,False
3,<NA>,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026,False
4,<NA>,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918,False


## Sanity check: matches `run_pipeline()` exactly

`run_pipeline()` is the single entry point `main.py` and the CLI app call —
confirming it reproduces the same result as the manual step-by-step walk
above is a useful check that nothing was left out.

In [9]:
df_check = pipeline.run_pipeline(pipeline.DEFAULT_INPUT, pipeline.DEFAULT_OUTPUT)
assert df_check.shape == df4.shape, "Shape mismatch between manual walkthrough and run_pipeline()!"
assert list(df_check.columns) == list(df4.columns), "Column mismatch!"
print(f"Confirmed: run_pipeline() output matches the step-by-step walkthrough above — shape {df_check.shape}.")

2026-09-15 11:46:25 | INFO     | pipeline | Successfully loaded raw data from Metro_Interstate_Traffic_Volume.csv: 48204 rows, 9 columns
2026-09-15 11:46:25 | INFO     | pipeline | Schema validation passed — all 9 expected columns present
2026-09-15 11:46:25 | WARNING  | pipeline | Standardised casing/whitespace on 1730 rows in 'weather_description' (reason: inconsistent capitalisation, e.g. 'Sky is Clear' vs 'sky is clear')
2026-09-15 11:46:25 | INFO     | pipeline | Standardised 'holiday' column: 61 rows flagged as a public holiday, 48143 rows are ordinary days (added boolean 'is_holiday' column)
2026-09-15 11:46:25 | INFO     | pipeline | Parsed date_time to datetime dtype for 48204 rows (from 48204 originally)
2026-09-15 11:46:25 | WARNING  | pipeline | Removed 17 fully duplicate rows (reason: identical row appeared more than once in the raw file)
2026-09-15 11:46:25 | WARNING  | pipeline | Collapsed 7612 rows that shared a timestamp with another row down to one row per hour (reaso